# Development of a Classifier for Sentiment Analysis in PySpark

## 1. Preprocessing

In [1]:
# Mount GDrive & Import dataset
#from google.colab import drive
#drive.mount("/content/gdrive")

#dataset_path = "/content/gdrive/MyDrive/data.csv"

In [2]:
from pyspark.sql import SparkSession

sc = SparkSession.builder.appName("FinancialSentimentAnalysis").getOrCreate()


In [ ]:
data = sc.read.csv("data.csv", header=True, inferSchema=True, quote='"', escape='"')
data.show(truncate=80)

+--------------------------------------------------------------------------------+---------+
|                                                                        Sentence|Sentiment|
+--------------------------------------------------------------------------------+---------+
|The GeoSolutions technology will leverage Benefon 's GPS solutions by providi...| positive|
|                         $ESI on lows, down $1.50 to $2.50 BK a real possibility| negative|
|For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m fro...| positive|
|According to the Finnish-Russian Chamber of Commerce , all the major construc...|  neutral|
|The Swedish buyout firm has sold its remaining 22.4 percent stake , almost ei...|  neutral|
|                                 $SPY wouldn't be surprised to see a green close| positive|
|                        Shell's $70 Billion BG Deal Meets Shareholder Skepticism| negative|
|SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANGE RELEASE OCTOBER 14 , 

In [4]:
data.printSchema()

root
 |-- Sentence: string (nullable = true)
 |-- Sentiment: string (nullable = true)



In [7]:
from pyspark.sql.functions import count, when, col

print("Missing values per column:")
data.select([count(when(col(c).isNull(), c)).alias(c) for c in data.columns]).show()

Missing values per column:
+--------+---------+
|Sentence|Sentiment|
+--------+---------+
|       0|        0|
+--------+---------+



In [8]:
data.groupBy("sentiment").count().show()

+---------+-----+
|sentiment|count|
+---------+-----+
| positive| 1852|
|  neutral| 3130|
| negative|  860|
+---------+-----+



In [9]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="Sentiment",
    outputCol="label"
)

data = label_indexer.fit(data).transform(data)
data.select("Sentiment", "label").show(10)

+---------+-----+
|Sentiment|label|
+---------+-----+
| positive|  1.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
|  neutral|  0.0|
| positive|  1.0|
| negative|  2.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
+---------+-----+
only showing top 10 rows


In [10]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Train count:", train_data.count())
print("Test count:", test_data.count())

Train count: 4725
Test count: 1117


In [11]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="Sentence",
    outputCol="tokens"
)

print("Tokens:")
tokenizer.transform(train_data).select("Sentence", "tokens").show(10, truncate=60)


Tokens:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                    Sentence|                                                      tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|#Apple breaks major support, here are some levels to watc...|[#apple, breaks, major, support,, here, are, some, levels...|
|#Apple up almost 20% from its February lows with very lit...|[#apple, up, almost, 20%, from, its, february, lows, with...|
|#FusionIQ NEW Positive Timing Signal on $SBUX Today https...|[#fusioniq, new, positive, timing, signal, on, $sbux, tod...|
|#LongPos $TSLA 256 Break-out thru 50 & 200- DMA (197-230)...|[#longpos, $tsla, 256, break-out, thru, 50, &, 200-, dma,...|
|#NDX component Tesla has announced it is recalling 2700 M...|[#ndx, component, tesla, has, announced, it, is, recallin...|


In [12]:
from pyspark.ml.feature import StopWordsRemover

stop_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

print("Filtered Tokens:")
stop_remover.transform(tokenizer.transform(train_data)).select("Sentence", "filtered_tokens").show(10, truncate=60)

Filtered Tokens:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                    Sentence|                                             filtered_tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|#Apple breaks major support, here are some levels to watc...|[#apple, breaks, major, support,, levels, watch, -, http:...|
|#Apple up almost 20% from its February lows with very lit...|[#apple, almost, 20%, february, lows, little, fanfare., $...|
|#FusionIQ NEW Positive Timing Signal on $SBUX Today https...|[#fusioniq, new, positive, timing, signal, $sbux, today, ...|
|#LongPos $TSLA 256 Break-out thru 50 & 200- DMA (197-230)...|[#longpos, $tsla, 256, break-out, thru, 50, &, 200-, dma,...|
|#NDX component Tesla has announced it is recalling 2700 M...|[#ndx, component, tesla, announced, recalling, 2700, 

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

# Initialize PorterStemmer once per Spark worker to reduce serialization overhead and prevent BrokenPipe/worker crashes
_stemmer = None

def stem_tokens(tokens):
    global _stemmer
    if not tokens:
        return []
    if _stemmer is None:
        import nltk
        from nltk.stem import PorterStemmer
        _stemmer = PorterStemmer()
    return [_stemmer.stem(token) for token in tokens]

stem_udf = udf(stem_tokens, ArrayType(StringType()))

# Register the UDF with the SparkSession so it can be used in an SQLTransformer pipeline later
sc.udf.register("stem_udf", stem_tokens, ArrayType(StringType()))

# Apply the stemming UDF to the filtered tokens
stemmed_data = stop_remover.transform(tokenizer.transform(train_data)).withColumn("stemmed_tokens", stem_udf(col("filtered_tokens")))
print("Stemmed Tokens:")
stemmed_data.select("Sentence", "stemmed_tokens").show(10, truncate=60)


Stemmed Tokens:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                    Sentence|                                              stemmed_tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|#Apple breaks major support, here are some levels to watc...|[#appl, break, major, support,, level, watch, -, http://s...|
|#Apple up almost 20% from its February lows with very lit...| [#appl, almost, 20%, februari, low, littl, fanfare., $aapl]|
|#FusionIQ NEW Positive Timing Signal on $SBUX Today https...|[#fusioniq, new, posit, time, signal, $sbux, today, https...|
|#LongPos $TSLA 256 Break-out thru 50 & 200- DMA (197-230)...|[#longpo, $tsla, 256, break-out, thru, 50, &, 200-, dma, ...|
|#NDX component Tesla has announced it is recalling 2700 M...|[#ndx, compon, tesla, announc, recal, 2700, model, x, 

In [14]:
import nltk
from pyspark.sql.functions import col

# Check if 'wordnet' corpus is already downloaded, if not, download it just once
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    print("Downloading 'wordnet' corpus...")
    nltk.download('wordnet', quiet=True)

_lemmatizer = None

def lemmatize_tokens(tokens):
    global _lemmatizer
    if not tokens:
        return []
    if _lemmatizer is None:
        import nltk
        from nltk.stem import WordNetLemmatizer
        # Download it on the driver, but workers might need it if running a distributed cluster
        try:
            nltk.data.find('corpora/wordnet')
        except:
            nltk.download('wordnet', quiet=True)
            nltk.download('omw-1.4', quiet=True)
        _lemmatizer = WordNetLemmatizer()
    return [_lemmatizer.lemmatize(token) for token in tokens]

lemmatize_udf = udf(lemmatize_tokens, ArrayType(StringType()))

# Register the UDF with the SparkSession so it can be used in an SQLTransformer pipeline later
sc.udf.register("lemmatize_udf", lemmatize_tokens, ArrayType(StringType()))

# Apply the lemmatization UDF to the filtered tokens
lemmatized_data = stop_remover.transform(tokenizer.transform(train_data)).withColumn("lemmatized_tokens", lemmatize_udf(col("filtered_tokens")))

print("Lemmatized Tokens:")
lemmatized_data.select("Sentence", "lemmatized_tokens").show(10, truncate=60)


Lemmatized Tokens:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                    Sentence|                                           lemmatized_tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|#Apple breaks major support, here are some levels to watc...|[#apple, break, major, support,, level, watch, -, http://...|
|#Apple up almost 20% from its February lows with very lit...|[#apple, almost, 20%, february, low, little, fanfare., $a...|
|#FusionIQ NEW Positive Timing Signal on $SBUX Today https...|[#fusioniq, new, positive, timing, signal, $sbux, today, ...|
|#LongPos $TSLA 256 Break-out thru 50 & 200- DMA (197-230)...|[#longpos, $tsla, 256, break-out, thru, 50, &, 200-, dma,...|
|#NDX component Tesla has announced it is recalling 2700 M...|[#ndx, component, tesla, announced, recalling, 2700

## 2. Building a Pipeline

In [15]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, CountVectorizer, IDF, Word2Vec, NGram
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import os
from contextlib import redirect_stderr

# 1. Base stages
text_processing_transformer = SQLTransformer(
    statement="SELECT *, stem_udf(filtered_tokens) AS stemmed_tokens, lemmatize_udf(filtered_tokens) AS lemmatized_tokens FROM __THIS__"
)
base_stages = [tokenizer, stop_remover, text_processing_transformer]

# 2. Define Feature Extraction variants for both sets of tokens
feature_variants = {}
for text_col in ["stemmed_tokens", "lemmatized_tokens"]:
    feature_variants[f"tfidf_{text_col}"] = [
        CountVectorizer(inputCol=text_col, outputCol="raw_features", vocabSize=5000, minDF=5.0),
        IDF(inputCol="raw_features", outputCol="features")
    ]
    feature_variants[f"bigram_tfidf_{text_col}"] = [
        NGram(n=2, inputCol=text_col, outputCol="bigrams"),
        CountVectorizer(inputCol="bigrams", outputCol="raw_features", vocabSize=5000, minDF=5.0),
        IDF(inputCol="raw_features", outputCol="features")
    ]
    feature_variants[f"word2vec_{text_col}"] = [
        Word2Vec(vectorSize=100, minCount=1, inputCol=text_col, outputCol="features")
    ]

# 3. Define Models and Hyperparameter Grids
lr = LogisticRegression(featuresCol="features", labelCol="label")
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)

grids = {
    "lr": (lr, ParamGridBuilder() \
        .addGrid(lr.maxIter, [100]) \
        .addGrid(lr.regParam, [0.01, 0.1]) \
        .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
        .build()),
    "dt": (dt, ParamGridBuilder() \
        .addGrid(dt.maxDepth, [8, 10]) \
        .addGrid(dt.minInstancesPerNode, [2, 5]) \
        .build()),
    "rf": (rf, ParamGridBuilder() \
        .addGrid(rf.numTrees, [100]) \
        .addGrid(rf.maxDepth, [10]) \
        .addGrid(rf.featureSubsetStrategy, ["sqrt"]) \
        .build())
}

# 4. Evaluators
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# 5. Run experiments with Pipelines and CrossValidator
results_table = []
best_overall_model = None
best_f1 = 0.0

# Using context manager to suppress standard error output (hides broken pipe crashes from pyspark daemon)
with open(os.devnull, 'w') as devnull, redirect_stderr(devnull):
    for feat_name, feat_stages in feature_variants.items():
        for model_name, (model, grid) in grids.items():
            print(f"Running pipeline + CrossValidator for: {feat_name} + {model_name}")

            # Build the full pipeline
            pipeline = Pipeline(stages=base_stages + feat_stages + [model])

            # Using CrossValidator. A practical alternative for big data is TrainValidationSplit
            cv = CrossValidator(estimator=pipeline,
                                estimatorParamMaps=grid,
                                evaluator=evaluator_f1,
                                numFolds=2,    # Reduced folds for demonstration speed
                                seed=42)

            # Fit on train_data
            cv_model = cv.fit(train_data)

            # Best model results on test_data
            predictions = cv_model.transform(test_data)
            f1_score = evaluator_f1.evaluate(predictions)
            accuracy = evaluator_acc.evaluate(predictions)

            print(f"  -> Best params yielded F1: {f1_score:.4f}, Accuracy: {accuracy:.4f}\n")

            results_table.append({
                "experiment": f"{feat_name}_{model_name}",
                "f1": f1_score,
                "accuracy": accuracy,
                "best_model": cv_model.bestModel
            })

            if f1_score > best_f1:
                best_f1 = f1_score
                best_overall_model = cv_model.bestModel


Running pipeline + CrossValidator for: tfidf_stemmed_tokens + lr
  -> Best params yielded F1: 0.7102, Accuracy: 0.7296

Running pipeline + CrossValidator for: tfidf_stemmed_tokens + dt
  -> Best params yielded F1: 0.6630, Accuracy: 0.7117

Running pipeline + CrossValidator for: tfidf_stemmed_tokens + rf
  -> Best params yielded F1: 0.5605, Accuracy: 0.6491

Running pipeline + CrossValidator for: bigram_tfidf_stemmed_tokens + lr
  -> Best params yielded F1: 0.5346, Accuracy: 0.6007

Running pipeline + CrossValidator for: bigram_tfidf_stemmed_tokens + dt
  -> Best params yielded F1: 0.4818, Accuracy: 0.5980

Running pipeline + CrossValidator for: bigram_tfidf_stemmed_tokens + rf
  -> Best params yielded F1: 0.4664, Accuracy: 0.5962

Running pipeline + CrossValidator for: word2vec_stemmed_tokens + lr
  -> Best params yielded F1: 0.5952, Accuracy: 0.6356

Running pipeline + CrossValidator for: word2vec_stemmed_tokens + dt
  -> Best params yielded F1: 0.5735, Accuracy: 0.6052

Running pipel

## 3. Results evaluation

In [16]:
# Output the best overall model
best_experiment = max(results_table, key=lambda x: x["f1"])

print("Best configuration:")
print("Experiment:", best_experiment["experiment"])
print(f"F1 Score: {best_experiment['f1']:.4f}")
print(f"Accuracy: {best_experiment['accuracy']:.4f}")
print("Pipeline Details:")
print(best_experiment["best_model"].stages)


Best configuration:
Experiment: tfidf_stemmed_tokens_lr
F1 Score: 0.7102
Accuracy: 0.7296
Pipeline Details:
[Tokenizer_664cfe1ef0b5, StopWordsRemover_97d7ff96cec6, SQLTransformer_c3168c7ccc42, CountVectorizerModel: uid=CountVectorizer_24fa60d63732, vocabularySize=1847, IDFModel: uid=IDF_737569b6ca80, numDocs=4725, numFeatures=1847, LogisticRegressionModel: uid=LogisticRegression_b45c855a389c, numClasses=3, numFeatures=1847]


In [17]:
import pandas as pd

# Creating a clean DataFrame containing the results
clean_results = [
    {"experiment": r["experiment"], "f1": r["f1"], "accuracy": r["accuracy"]}
    for r in results_table
]

results_df = pd.DataFrame(clean_results)
results_df = results_df.sort_values(by="f1", ascending=False)
results_df


,experiment,f1,accuracy
0,tfidf_stemmed_tokens_lr,0.710217,0.729633
9,tfidf_lemmatized_tokens_lr,0.709812,0.730528
1,tfidf_stemmed_tokens_dt,0.663048,0.711728
10,tfidf_lemmatized_tokens_dt,0.653028,0.703671
17,word2vec_lemmatized_tokens_rf,0.598961,0.634736
16,word2vec_lemmatized_tokens_dt,0.597823,0.635631
6,word2vec_stemmed_tokens_lr,0.595168,0.635631
8,word2vec_stemmed_tokens_rf,0.589913,0.628469
15,word2vec_lemmatized_tokens_lr,0.587839,0.631155
7,word2vec_stemmed_tokens_dt,0.573503,0.605192


## Interpretation of results

The best result on the Financial Sentiment Analysis dataset was achieved by **Logistic Regression with TF-IDF representation**.
Although the preprocessing methods had very similar results, the variant with the **Stemming** technique (tfidf_stemmed_tokens_lr) gave a slightly better F1 Score (≈ 0.71), while **Lemmatization** (tfidf_lemmatized_tokens_lr) achieved a slightly higher correct class recognition, i.e., Accuracy (≈ 0.73).

Key observations:
1. **Preprocessing (Stemming vs Lemmatization):** Both techniques produce practically the same level of performance. In this specific case, the more aggressive shortening of words (Stemming) effectively groups the basic meaning of sentiment just as well as more sophisticated lemmatization.
2. **Feature Extraction:** TF-IDF is dramatically more successful than simple sparse counting. It proved significantly better than Word2Vec (which requires a much larger word base to achieve solid values) and extremely better than bigrams, confirming that key individual financial words (unigrams) are the strongest indicators of sentiment.
3. **Classification Models:** Linear Logistic Regression dominates. Random Forest and Decision Tree suffer in high-dimensional vector spaces created by textual data and have a drastically worse F1 score (≈ 0.65 for DT and ≈ 0.56 for RF with TF-IDF).

## Conclusion

The winning pipeline approach is **TF-IDF + Logistic Regression**. Due to the lower consumption of computational resources during processing, Stemming may be preferred over Lemmatization because it requires simpler dependencies alongside a practically identical final result, stability, and accuracy that lie in the fact that patterns in positive and negative news behave almost linearly separable in a vast text frequency space.